# DME-Encoder Scenario Module - Kaggle Notebook

This notebook reuses a trained forecast checkpoint to produce constrained aggregate future-behavior scenarios and evaluate scenario metrics against validation/test future suffixes.

In [ ]:
# Cell 1 - Setup & Install
import json
import math
import pathlib
import pickle
import subprocess
import sys
import warnings

warnings.filterwarnings("ignore")

WORKING_DIR = pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle/working").exists() else pathlib.Path.cwd()
REPO_CANDIDATES = [
    WORKING_DIR / "denoising-event-sequences",
    pathlib.Path.cwd(),
    pathlib.Path.cwd().parent,
]
REPO_DIR = next(
    (p for p in REPO_CANDIDATES if (p / "src").exists() and (p / "scripts").exists()),
    REPO_CANDIDATES[0],
)
if not REPO_DIR.exists():
    raise FileNotFoundError(f"Repository not found. Checked: {REPO_CANDIDATES}")

subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR), "--quiet"], check=True)
sys.path.insert(0, str(REPO_DIR))

required_imports = {
    "yaml": "pyyaml",
    "pyarrow": "pyarrow",
    "sklearn": "scikit-learn",
}
missing_packages = []
for module_name, package_name in required_imports.items():
    try:
        __import__(module_name)
    except ImportError:
        missing_packages.append(package_name)
if missing_packages:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing_packages, "--quiet"], check=True)

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import DataLoader

print(f"Repo       : {REPO_DIR}")
print(f"Working dir: {WORKING_DIR}")
print(f"Torch      : {torch.__version__}")

In [ ]:
# Cell 2 - Paths & Runtime Knobs
DATA_ROOT = WORKING_DIR / "data" if (WORKING_DIR / "data").exists() else REPO_DIR / "data"
PROCESSED_DIR = DATA_ROOT / "processed" / "rosbank_forecast"
OUTPUT_DIR = WORKING_DIR / "outputs" if (WORKING_DIR / "outputs").exists() else REPO_DIR / "outputs"
FORECAST_OUTPUT_DIR = OUTPUT_DIR / "forecast_pretrain"
SCENARIO_OUTPUT_DIR = OUTPUT_DIR / "scenario_module"
LOG_DIR = OUTPUT_DIR / "logs"

CONFIG_CANDIDATES = [
    WORKING_DIR / "forecast_pipeline.yaml",
    FORECAST_OUTPUT_DIR / "checkpoints" / "forecast_pretrain_config.yaml",
    REPO_DIR / "configs" / "base.yaml",
]
CONFIG_PATH = next((p for p in CONFIG_CANDIDATES if p.exists()), CONFIG_CANDIDATES[-1])

RUN_FORECAST_PRETRAIN_IF_MISSING = False
BATCH_SIZE = 128
TOP_K = 5
NUM_SCENARIO_EXAMPLES = 50
EVAL_SPLITS = ["val", "test"]

SCENARIO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

def run_logged(cmd: list[str], log_path: pathlib.Path) -> None:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("Running:", " ".join(map(str, cmd)))
    with log_path.open("w") as f:
        proc = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, text=True)
    if proc.returncode != 0:
        tail = log_path.read_text(errors="ignore")[-4000:]
        raise RuntimeError(f"Command failed with code {proc.returncode}. Log tail:\n{tail}")

print(f"Config             : {CONFIG_PATH}")
print(f"Processed data dir : {PROCESSED_DIR}")
print(f"Forecast output dir: {FORECAST_OUTPUT_DIR}")
print(f"Scenario output dir: {SCENARIO_OUTPUT_DIR}")

In [ ]:
# Cell 3 - Imports
from src.data.collate import collate_fn
from src.data.dataset import EventSequenceDataset
from src.data.forecasting import (
    get_num_feature_dim,
    load_forecast_stats,
    scenario_from_outputs,
    scenario_from_targets,
)
from src.data.splits import load_splits
from src.evaluation.forecasting import (
    compute_forecast_baseline_metrics,
    compute_forecast_quality_metrics,
)
from src.models.dme_encoder import DMEEncoder
from src.utils.config import load_experiment_config
from src.utils.seed import get_device, set_seed

device = get_device()
print(f"Device: {device}")

In [ ]:
# Cell 4 - Discover Forecast Artifacts
def resolve_artifact_path(value: str | pathlib.Path, bases: list[pathlib.Path]) -> pathlib.Path:
    path = pathlib.Path(value)
    if path.is_absolute() and path.exists():
        return path
    if path.exists():
        return path.resolve()
    for base in bases:
        candidate = base / path
        if candidate.exists():
            return candidate.resolve()
    return path

def find_forecast_summary() -> pathlib.Path | None:
    metrics_dir = FORECAST_OUTPUT_DIR / "metrics"
    preferred = metrics_dir / f"{CONFIG_PATH.stem}_forecast_pretrain_summary.json"
    if preferred.exists():
        return preferred
    candidates = sorted(
        metrics_dir.glob("*_forecast_pretrain_summary.json"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    return candidates[0] if candidates else None

FORECAST_SUMMARY_PATH = find_forecast_summary()
if FORECAST_SUMMARY_PATH is None and RUN_FORECAST_PRETRAIN_IF_MISSING:
    cmd = [
        sys.executable,
        str(REPO_DIR / "scripts" / "forecast_pretrain.py"),
        "--config",
        str(CONFIG_PATH),
        "--data-dir",
        str(PROCESSED_DIR),
        "--output-dir",
        str(FORECAST_OUTPUT_DIR),
    ]
    run_logged(cmd, LOG_DIR / "scenario_forecast_pretrain.log")
    FORECAST_SUMMARY_PATH = find_forecast_summary()

if FORECAST_SUMMARY_PATH is None:
    raise FileNotFoundError(
        "Forecast summary not found. Run kaggle_forecast_pipeline first or set "
        "RUN_FORECAST_PRETRAIN_IF_MISSING=True."
    )

forecast_summary = json.loads(FORECAST_SUMMARY_PATH.read_text())
FORECAST_CKPT_PATH = resolve_artifact_path(
    forecast_summary["best_checkpoint"],
    [WORKING_DIR, OUTPUT_DIR, FORECAST_OUTPUT_DIR, REPO_DIR],
)
FORECAST_STATS_PATH = resolve_artifact_path(
    forecast_summary["forecast_stats"],
    [WORKING_DIR, OUTPUT_DIR, FORECAST_OUTPUT_DIR, REPO_DIR],
)

assert FORECAST_CKPT_PATH.exists(), f"Missing checkpoint: {FORECAST_CKPT_PATH}"
assert FORECAST_STATS_PATH.exists(), f"Missing forecast stats: {FORECAST_STATS_PATH}"
assert (PROCESSED_DIR / "events.parquet").exists(), f"Missing events.parquet in {PROCESSED_DIR}"
assert (PROCESSED_DIR / "splits.json").exists(), f"Missing splits.json in {PROCESSED_DIR}"
assert (PROCESSED_DIR / "preprocessor.pkl").exists(), f"Missing preprocessor.pkl in {PROCESSED_DIR}"

print(f"Forecast summary   : {FORECAST_SUMMARY_PATH}")
print(f"Forecast checkpoint: {FORECAST_CKPT_PATH}")
print(f"Forecast stats     : {FORECAST_STATS_PATH}")

In [ ]:
# Cell 5 - Load Data, Config, and Stats
set_seed(42)

checkpoint = torch.load(FORECAST_CKPT_PATH, map_location="cpu", weights_only=False)
config = checkpoint.get("config") or load_experiment_config(base_path=str(CONFIG_PATH))
config = {
    **config,
    "forecasting": {**config.get("forecasting", {}), "enabled": True, "scenario_top_k": TOP_K},
}

with (PROCESSED_DIR / "preprocessor.pkl").open("rb") as f:
    prep = pickle.load(f)
splits = load_splits(PROCESSED_DIR / "splits.json")
df_events = pd.read_parquet(PROCESSED_DIR / "events.parquet")
forecast_stats = load_forecast_stats(FORECAST_STATS_PATH)

vocab_info = forecast_summary.get("vocab_info") or checkpoint.get("vocab_info") or {
    "event_type_vocab_size": len(prep.vocab.get(prep.event_type_col, {})),
    "cat_vocab_sizes": [len(prep.vocab.get(col, {})) for col in prep.categorical_cols],
    "num_num_features": get_num_feature_dim(prep, config),
    "num_classes": int(config.get("training", {}).get("num_classes", 2)),
}

print(
    "Data loaded | rows=%d | train=%d val=%d test=%d"
    % (len(df_events), len(splits.get("train", [])), len(splits.get("val", [])), len(splits.get("test", [])))
)
print("Vocab info:", {k: v for k, v in vocab_info.items() if k != "cat_vocab_sizes"})
print("Cat vocab sizes:", vocab_info.get("cat_vocab_sizes", []))

In [ ]:
# Cell 6 - Build Forecast DataLoaders
def make_forecast_loader(split_name: str) -> DataLoader | None:
    entity_ids = splits.get(split_name, [])
    if not entity_ids:
        print(f"Split '{split_name}' is empty or missing; skipping")
        return None
    dataset = EventSequenceDataset(
        df_events,
        entity_ids,
        prep,
        config,
        mode="forecast",
        forecast_stats=forecast_stats,
    )
    if len(dataset) == 0:
        print(f"Split '{split_name}' has no valid forecast cuts; skipping")
        return None
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
    )

loaders = {split: make_forecast_loader(split) for split in EVAL_SPLITS}
loaders = {split: loader for split, loader in loaders.items() if loader is not None}
if not loaders:
    raise RuntimeError("No non-empty forecast evaluation loaders were built")

for split, loader in loaders.items():
    print(f"{split:>5s}: {len(loader.dataset)} entities | {len(loader)} batches")

In [ ]:
# Cell 7 - Load Forecast Model
model = DMEEncoder(config, vocab_info)
state = checkpoint["model_state_dict"]
forecast_head_prefixes = [
    "future_event_type_profile_head",
    "future_count_bucket_head",
    "future_amount_stats_head",
    "future_gap_bucket_head",
    "future_cat_profile_head",
]
missing_prefixes = [
    prefix for prefix in forecast_head_prefixes if not any(k.startswith(prefix) for k in state)
]
if missing_prefixes:
    raise RuntimeError(
        "Checkpoint does not contain forecast heads required for scenario inference: "
        + ", ".join(missing_prefixes)
    )

missing_keys, unexpected_keys = model.load_state_dict(state, strict=False)
missing_forecast_keys = [k for k in missing_keys if k.startswith("future_")]
if missing_forecast_keys:
    raise RuntimeError(f"Forecast head weights were not loaded: {missing_forecast_keys}")
if unexpected_keys:
    print("Unexpected checkpoint keys ignored:", unexpected_keys[:10])

model.to(device)
model.eval()
print("Forecast model loaded")

In [ ]:
# Cell 8 - Scenario Inference Helpers
def batch_to_device(value, target_device):
    if isinstance(value, torch.Tensor):
        return value.to(target_device)
    if isinstance(value, dict):
        return {k: batch_to_device(v, target_device) for k, v in value.items()}
    if isinstance(value, list):
        return [batch_to_device(v, target_device) for v in value]
    return value

def slice_outputs(outputs: dict, index: int) -> dict:
    sliced = {}
    for key, value in outputs.items():
        if isinstance(value, torch.Tensor):
            sliced[key] = value[index : index + 1]
        elif isinstance(value, list):
            sliced[key] = [item[index : index + 1] for item in value]
        else:
            sliced[key] = value
    return sliced

def add_metric_sums(metric_sums: dict, metrics: dict) -> None:
    for key, value in metrics.items():
        value = float(value)
        if math.isfinite(value):
            metric_sums[key] = metric_sums.get(key, 0.0) + value

def collect_split(split_name: str, loader: DataLoader) -> tuple[dict, list[dict]]:
    metric_sums: dict[str, float] = {}
    rows: list[dict] = []
    n_batches = 0
    with torch.no_grad():
        for batch in loader:
            batch_on_device = batch_to_device(batch, device)
            outputs = model(batch_on_device, mode="forecast")
            targets = batch_on_device["forecast_targets"]
            metrics = compute_forecast_quality_metrics(outputs, targets, forecast_stats, top_k=TOP_K)
            baseline_metrics = compute_forecast_baseline_metrics(
                batch_on_device,
                targets,
                forecast_stats,
                top_k=TOP_K,
            )
            add_metric_sums(metric_sums, {**metrics, **baseline_metrics})
            n_batches += 1

            for i, entity_id in enumerate(batch["entity_id"]):
                rows.append(
                    {
                        "split": split_name,
                        "entity_id": entity_id,
                        "predicted_scenario": scenario_from_outputs(
                            slice_outputs(outputs, i),
                            forecast_stats,
                            prep.vocab,
                            top_k=TOP_K,
                        ),
                        "true_future_summary": scenario_from_targets(
                            targets,
                            forecast_stats,
                            prep.vocab,
                            index=i,
                            top_k=TOP_K,
                        ),
                    }
                )

    metrics = {key: value / max(1, n_batches) for key, value in metric_sums.items()}
    metrics["num_entities"] = len(rows)
    metrics["num_batches"] = n_batches
    return metrics, rows

In [ ]:
# Cell 9 - Run Scenario Evaluation
metric_rows: list[dict] = []
scenario_rows: list[dict] = []

for split_name, loader in loaders.items():
    metrics, rows = collect_split(split_name, loader)
    metric_rows.append({"split": split_name, **metrics})
    scenario_rows.extend(rows)
    print(f"{split_name}: collected {len(rows)} scenario rows")

metrics_df = pd.DataFrame(metric_rows)
metric_cols = [c for c in metrics_df.columns if c not in {"split"}]
display(metrics_df[["split", *metric_cols]])

In [ ]:
# Cell 10 - Metric Comparison Table
MODEL_METRICS = [
    "future_count_accuracy",
    "future_count_macro_f1",
    "future_gap_accuracy",
    "future_gap_macro_f1",
    "event_profile_cosine",
    "event_profile_mae",
    "event_profile_top5_recall",
    "amount_stats_mae",
    "cat_profile_cosine_mean",
    "cat_profile_mae_mean",
]

comparison_rows = []
for row in metric_rows:
    split_name = row["split"]
    for label, prefix in [
        ("model", ""),
        ("global_frequency_baseline", "baseline_global_"),
        ("prefix_frequency_baseline", "baseline_prefix_"),
    ]:
        out = {"split": split_name, "system": label}
        for metric in MODEL_METRICS:
            out[metric] = row.get(prefix + metric)
        comparison_rows.append(out)

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

In [ ]:
# Cell 11 - Save Scenario Artifacts
def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [json_safe(v) for v in value]
    if isinstance(value, tuple):
        return [json_safe(v) for v in value]
    if isinstance(value, pathlib.Path):
        return str(value)
    if isinstance(value, np.generic):
        return json_safe(value.item())
    if isinstance(value, float):
        if not math.isfinite(value):
            raise ValueError(f"Non-finite metric/value encountered: {value}")
        return value
    return value

scenario_examples = scenario_rows[:NUM_SCENARIO_EXAMPLES]
SCENARIO_EVAL_METRICS_PATH = SCENARIO_OUTPUT_DIR / "scenario_eval_metrics.json"
SCENARIO_EXAMPLES_PATH = SCENARIO_OUTPUT_DIR / "scenario_examples.json"
SCENARIO_PREDICTIONS_PATH = SCENARIO_OUTPUT_DIR / "scenario_predictions.parquet"
SCENARIO_COMPARISON_PATH = SCENARIO_OUTPUT_DIR / "scenario_metric_comparison.csv"

SCENARIO_EVAL_METRICS_PATH.write_text(
    json.dumps(
        json_safe(
            {
                "forecast_summary": FORECAST_SUMMARY_PATH,
                "forecast_checkpoint": FORECAST_CKPT_PATH,
                "forecast_stats": FORECAST_STATS_PATH,
                "metrics": metric_rows,
                "comparison": comparison_rows,
            }
        ),
        indent=2,
        ensure_ascii=False,
    )
)
SCENARIO_EXAMPLES_PATH.write_text(
    json.dumps(json_safe(scenario_examples), indent=2, ensure_ascii=False)
)
comparison_df.to_csv(SCENARIO_COMPARISON_PATH, index=False)

predictions_df = pd.DataFrame(scenario_rows)
predictions_export_df = predictions_df.assign(
    predicted_scenario_json=predictions_df["predicted_scenario"].map(
        lambda x: json.dumps(json_safe(x), ensure_ascii=False)
    ),
    true_future_summary_json=predictions_df["true_future_summary"].map(
        lambda x: json.dumps(json_safe(x), ensure_ascii=False)
    ),
).drop(columns=["predicted_scenario", "true_future_summary"])
predictions_export_df.to_parquet(SCENARIO_PREDICTIONS_PATH, index=False)

print("Scenario artifacts saved:")
print(" -", SCENARIO_EVAL_METRICS_PATH)
print(" -", SCENARIO_EXAMPLES_PATH)
print(" -", SCENARIO_PREDICTIONS_PATH)
print(" -", SCENARIO_COMPARISON_PATH)

In [ ]:
# Cell 12 - Inspect Scenario Examples
def compact_scenario_for_display(row: dict) -> dict:
    pred = row["predicted_scenario"]
    true = row["true_future_summary"]
    return {
        "split": row["split"],
        "entity_id": row["entity_id"],
        "pred_count": pred["future_count_bucket"],
        "true_count": true["future_count_bucket"],
        "pred_gap": pred["gap_bucket"],
        "true_gap": true["gap_bucket"],
        "pred_event_types": ", ".join(str(x["token"]) for x in pred["dominant_event_types"][:TOP_K]),
        "true_event_types": ", ".join(str(x["token"]) for x in true["dominant_event_types"][:TOP_K]),
        "pred_positive_share": pred["expected_amount_stats"]["positive_share"],
        "true_positive_share": true["expected_amount_stats"]["positive_share"],
    }

example_display_df = pd.DataFrame(
    [compact_scenario_for_display(row) for row in scenario_examples[:20]]
)
display(example_display_df)

print("Detailed first scenario example:")
print(json.dumps(json_safe(scenario_examples[0]), indent=2, ensure_ascii=False) if scenario_examples else "No examples")